# Matbench Experimental Band Gap: Cross-Task Validation

This notebook extends the project beyond the initial `matbench_steels` validation. The goal is to check whether the workflow and modeling conclusions transfer to a second Matbench composition task.

Task: predict experimental band gap from composition using `matbench_expt_gap`.

## Why this notebook matters

The first notebook showed that Extra Trees improved over a Random Forest baseline on steel yield strength. A single-task result is not enough for a strong project. This notebook tests the same style of workflow on a larger composition-only Matbench task:

- Same benchmark protocol: official Matbench folds.
- Same featurization family: Magpie composition descriptors.
- Similar model comparison: linear baseline, Random Forest, Extra Trees, boosted trees, and a simple voting ensemble.

This gives the project a more defensible story: not just one result, but a small benchmark study.

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matbench.bench import MatbenchBenchmark
from matminer.featurizers.composition import ElementProperty
from pymatgen.core import Composition
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor, VotingRegressor
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=UserWarning)

CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
RESULTS_DIR = PROJECT_ROOT / 'results'
METRICS_DIR = RESULTS_DIR / 'metrics'
FIGURES_DIR = RESULTS_DIR / 'figures'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
for directory in [METRICS_DIR, FIGURES_DIR, PROCESSED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
TASK_NAME = 'matbench_expt_gap'
print('Project root:', PROJECT_ROOT)

## Load Matbench task

`matbench_expt_gap` predicts experimentally measured band gaps in eV from composition. It is larger than `matbench_steels`, but still small enough for classical tabular models.

In [ ]:
benchmark = MatbenchBenchmark(autoload=False, subset=[TASK_NAME])
task = list(benchmark.tasks)[0]
task.load()

print('Task metadata:')
for key in ['input_type', 'task_type', 'target', 'unit', 'n_samples']:
    value = task.metadata.get(key, task.metadata.get('num_entries'))
    print(f'  {key}: {value}')
print('Official folds:', task.folds_nums)
task.df.head()

## Featurize compositions

We use the same Magpie composition descriptors as the steels notebook so the two task results are directly comparable. The computed feature matrix is cached under `data/processed/` because this task has 4604 compositions.

In [ ]:
feature_cache = PROCESSED_DIR / f'{TASK_NAME}_magpie_features.csv'


def featurize_magpie(formulas: pd.Series, cache_path: Path) -> pd.DataFrame:
    if cache_path.exists():
        features = pd.read_csv(cache_path, index_col=0)
        features.index = formulas.index
        return features

    compositions = formulas.map(Composition)
    feature_input = pd.DataFrame({'composition': compositions}, index=formulas.index)
    featurizer = ElementProperty.from_preset('magpie')
    features = featurizer.featurize_dataframe(
        feature_input,
        col_id='composition',
        ignore_errors=False,
        inplace=False,
        pbar=True,
    )
    features = features.drop(columns=['composition'])
    features = features.apply(pd.to_numeric, errors='coerce')
    features = features.replace([np.inf, -np.inf], np.nan)
    features.to_csv(cache_path)
    return features

X = featurize_magpie(task.df['composition'], feature_cache)
y = task.df['gap expt']
print('Feature matrix:', X.shape)
print('Target:', y.shape)
X.iloc[:5, :8]

## Model evaluation helper

As in the steels notebook, every model is evaluated on the official Matbench folds. Imputation and scaling are fit only on the training part of each fold.

In [ ]:
def evaluate_estimator(
    estimator,
    X_all: pd.DataFrame,
    task,
    *,
    model_label: str,
    scale_features: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    metrics = []
    predictions = []

    for fold in task.folds_nums:
        X_train_raw, y_train = task.get_train_and_val_data(fold)
        X_test_raw, y_test = task.get_test_data(fold, include_target=True)

        X_train = X_all.loc[X_train_raw.index]
        X_test = X_all.loc[X_test_raw.index]

        steps = [('imputer', SimpleImputer(strategy='median'))]
        if scale_features:
            steps.append(('scaler', StandardScaler()))
        steps.append(('model', clone(estimator)))

        model = Pipeline(steps=steps)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        metrics.append({
            'task': TASK_NAME,
            'feature_set': 'magpie',
            'model': model_label,
            'fold': fold,
            'train_size': len(X_train),
            'test_size': len(X_test),
            'n_features': X_train.shape[1],
            'mae': mean_absolute_error(y_test, y_pred),
            'r2': r2_score(y_test, y_pred),
        })

        predictions.append(pd.DataFrame({
            'model': model_label,
            'fold': fold,
            'mbid': y_test.index,
            'y_true': y_test.to_numpy(),
            'y_pred': y_pred,
            'absolute_error': np.abs(y_test.to_numpy() - y_pred),
        }))

    return pd.DataFrame(metrics), pd.concat(predictions, ignore_index=True)

## Run model comparison

The tree models use 300 estimators here to keep runtime reasonable on a laptop while still giving stable ensemble behavior.

In [ ]:
rf_estimator = RandomForestRegressor(
    n_estimators=300,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
extra_trees_estimator = ExtraTreesRegressor(
    n_estimators=300,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
hgb_estimator = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.04,
    l2_regularization=0.01,
    random_state=RANDOM_SEED,
)
ensemble_estimator = VotingRegressor([
    ('rf', rf_estimator),
    ('extra_trees', extra_trees_estimator),
    ('hgb', hgb_estimator),
])

model_specs = [
    ('dummy_mean_baseline', DummyRegressor(strategy='mean'), False),
    ('ridge_cv_all_magpie', RidgeCV(alphas=np.logspace(-3, 3, 13)), True),
    ('random_forest_all_magpie', rf_estimator, False),
    ('extra_trees_all_magpie', extra_trees_estimator, False),
    ('hist_gradient_boosting_all_magpie', hgb_estimator, False),
    ('rf_extra_trees_hgb_voting_ensemble', ensemble_estimator, False),
]

metric_tables = []
prediction_tables = []
for model_label, estimator, scale_features in model_specs:
    print('Running', model_label)
    metrics_i, predictions_i = evaluate_estimator(
        estimator,
        X,
        task,
        model_label=model_label,
        scale_features=scale_features,
    )
    metric_tables.append(metrics_i)
    prediction_tables.append(predictions_i)

model_metrics = pd.concat(metric_tables, ignore_index=True)
model_predictions = pd.concat(prediction_tables, ignore_index=True)
model_metrics

In [ ]:
model_summary = (
    model_metrics
    .groupby('model')
    .agg(
        mean_mae=('mae', 'mean'),
        std_mae=('mae', 'std'),
        mean_r2=('r2', 'mean'),
        n_features=('n_features', 'mean'),
    )
    .sort_values('mean_mae')
    .reset_index()
)
model_summary

## Plot results

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_df = model_summary.sort_values('mean_mae')
ax.barh(plot_df['model'], plot_df['mean_mae'], xerr=plot_df['std_mae'], alpha=0.85)
ax.set_title('Model comparison on matbench_expt_gap')
ax.set_xlabel('Mean MAE across official folds (eV)')
ax.invert_yaxis()
plt.tight_layout()
figure_path = FIGURES_DIR / 'notebook_expt_gap_model_comparison.png'
plt.savefig(figure_path, dpi=200)
plt.show()
print('Saved figure:', figure_path)

In [ ]:
best_model_name = model_summary.iloc[0]['model']
best_predictions = model_predictions[model_predictions['model'] == best_model_name]

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(best_predictions['y_true'], best_predictions['y_pred'], alpha=0.35, s=16)
min_val = min(best_predictions['y_true'].min(), best_predictions['y_pred'].min())
max_val = max(best_predictions['y_true'].max(), best_predictions['y_pred'].max())
ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1)
ax.set_title(f'Parity plot: {best_model_name}')
ax.set_xlabel('Measured band gap (eV)')
ax.set_ylabel('Predicted band gap (eV)')
plt.tight_layout()
figure_path = FIGURES_DIR / 'notebook_expt_gap_best_model_parity.png'
plt.savefig(figure_path, dpi=200)
plt.show()
print('Saved figure:', figure_path)

## Save results

In [ ]:
metrics_path = METRICS_DIR / 'notebook_expt_gap_model_comparison.csv'
summary_path = METRICS_DIR / 'notebook_expt_gap_model_summary.csv'
predictions_path = RESULTS_DIR / 'predictions' / 'notebook_expt_gap_model_predictions.csv'
predictions_path.parent.mkdir(parents=True, exist_ok=True)

model_metrics.to_csv(metrics_path, index=False)
model_summary.to_csv(summary_path, index=False)
model_predictions.to_csv(predictions_path, index=False)

print('Saved fold metrics:', metrics_path)
print('Saved model summary:', summary_path)
print('Saved predictions:', predictions_path)
model_summary

## Preliminary conclusion

This second task turns the project into a small benchmark instead of a one-off result. Compare the best `matbench_expt_gap` model against the steels notebook:

- If Extra Trees is also strong here, the project can argue that randomized tree ensembles are a robust classical baseline for Magpie features.
- If a different model wins, the project can discuss task dependence and why a single method may not dominate all small materials datasets.
- After TabPFN authorization, the same fold-based comparison can be repeated with `TabPFNRegressor` on the smaller steels task first.